# LAK-7 — Partitioning, hidden partitioning & evolution

**Break -> Detect -> Fix -> Prove.** Two Iceberg features that Hive-style partitioning can't match:

1. **Hidden partitioning** — Iceberg derives the partition value from a *transform* of a real
   column (`days(ts)`), so you prune by filtering the **raw `ts`** column. No separate
   `event_date` column, and no `CAST(ts AS DATE)` in your `WHERE` to silently break pruning.
2. **Partition evolution** — change the partition layout (`ADD`/`DROP PARTITION FIELD`) as a
   **metadata-only** operation that touches only **new** data; old files keep their old spec and
   are never rewritten.

**Pre-requisite:** the unified Spark server is up (`make up`) -> Spark UI at http://localhost:4040.
This notebook connects via Spark Connect. **Laptop-safe:** a few dozen rows spanning a handful of
days, all under `.tmp/`; the table is dropped at the end (`make clean` clears the rest).

See the companion [`README.md`](./README.md), the [Spark-UI guide](../../docs/spark-ui-guide.md),
and the [troubleshooting sheet](../../docs/troubleshooting.md) (LAK-7 row). Related: **SPK-7**
(partition pruning / predicate pushdown) and **LAK-8** (per-partition MERGE cost).

In [ ]:
from common.spark_session import spark, display_df
from common.iceberg_meta import table_health
from pyspark.sql import functions as F

T = "iceberg_catalog.default.lak7_events"
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 — create a table with **hidden partitioning** (`days(ts)`)

`days(ts)` is a **partition transform**: Iceberg derives the partition value from the `ts`
timestamp and records that mapping in table metadata. There is **no** `event_date` column in the
schema — the partitioning is *hidden*. We then load a few rows across **four days** (tiny data —
the point is the layout, not the volume), one `INSERT` per day so each day is its own snapshot.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
spark.sql(f"""
    CREATE TABLE {T} (
        event_id    BIGINT,
        customer_id BIGINT,
        amount      DOUBLE,
        ts          TIMESTAMP
    ) USING iceberg
    PARTITIONED BY (days(ts))
""")

# A few events per day across 2026-06-01 .. 2026-06-04 (one INSERT per day = one snapshot/day).
for vals in [
    "(1,101,12.0,TIMESTAMP'2026-06-01 09:00:00'),(2,102,20.0,TIMESTAMP'2026-06-01 14:30:00'),(3,103,5.0,TIMESTAMP'2026-06-01 18:05:00')",
    "(4,101,8.5,TIMESTAMP'2026-06-02 10:15:00'),(5,104,42.0,TIMESTAMP'2026-06-02 16:40:00')",
    "(6,102,17.0,TIMESTAMP'2026-06-03 08:20:00'),(7,105,3.0,TIMESTAMP'2026-06-03 12:00:00'),(8,103,9.0,TIMESTAMP'2026-06-03 21:45:00')",
    "(9,101,25.0,TIMESTAMP'2026-06-04 11:30:00'),(10,106,1.5,TIMESTAMP'2026-06-04 19:10:00')",
]:
    spark.sql(f"INSERT INTO {T} VALUES {vals}")

print("rows loaded:", spark.table(T).count(), "  schema (NO event_date column):", spark.table(T).columns)

## 1. Detect it — hidden partitioning prunes on the **raw** `ts`

Query a single day with a predicate on **`ts` itself** — no `event_date`, no `CAST(ts AS DATE)`.
Iceberg matches the predicate against the `days(ts)` transform and prunes the other partitions at
**planning time**. `df.explain()` shows the pushed filter on `ts` (look for the Iceberg
`BatchScan` and the pushed `PushedFilters` / partition filter on `ts`, plus the file count in the
scan).

In [ ]:
one_day = spark.sql(f"""
    SELECT * FROM {T}
    WHERE ts >= TIMESTAMP'2026-06-03 00:00:00'
      AND ts <  TIMESTAMP'2026-06-04 00:00:00'
""")

print("rows in 2026-06-03:", one_day.count())
print("\n--- physical plan (note the pushed filter on `ts`; no event_date / CAST needed) ---\n")
one_day.explain()

## 2. The partition layout — `<table>.partitions`

Iceberg's `.partitions` metadata table shows one row per partition: the `days(ts)` value plus its
`record_count` / `file_count`. This is the physical grouping the planner pruned against above —
four days loaded, four partitions, **no `event_date` column anywhere**.

In [ ]:
spark.sql(f"SELECT partition, spec_id, record_count, file_count FROM {T}.partitions ORDER BY partition").show(truncate=False)

**Prove the pruning numerically.** A day-scoped query reads only that day's files; a
full-table scan reads them all. We compare `inputFiles()` (files the planner selected) and rows
for each — fewer files/rows for the pruned query is hidden partitioning doing its job.

In [ ]:
full_files  = len(spark.table(T).inputFiles())
day_files   = len(one_day.inputFiles())
full_rows   = spark.table(T).count()
day_rows    = one_day.count()

print(f"full-table scan : {full_files} files, {full_rows} rows")
print(f"one-day (pruned): {day_files} files, {day_rows} rows   <- pruned via days(ts) on raw `ts`")
print(f"\npruned away    : {full_files - day_files} files / {full_rows - day_rows} rows not scanned")

## 3. Fix it (evolve) — `ADD PARTITION FIELD bucket(8, customer_id)`

The access pattern now also looks up a **single customer's** events; `days(ts)` alone doesn't help
that. **Add** a `bucket(8, customer_id)` partition field — a **metadata-only** change (no rewrite).
Then insert more rows and watch the new files adopt the new spec while the old files keep theirs.

In [ ]:
health_before = table_health(spark, T, "before evolution")

spark.sql(f"ALTER TABLE {T} ADD PARTITION FIELD bucket(8, customer_id)")

# New rows on a NEW day, written under the evolved spec (days(ts) + bucket(8, customer_id)).
spark.sql(f"""INSERT INTO {T} VALUES
    (11,101,30.0,TIMESTAMP'2026-06-05 09:00:00'),(12,107,4.0,TIMESTAMP'2026-06-05 13:20:00'),
    (13,102,11.0,TIMESTAMP'2026-06-05 17:50:00'),(14,108,6.5,TIMESTAMP'2026-06-05 22:05:00')
""")

health_after = table_health(spark, T, "after evolution")
print("partition spec evolved; 4 new rows inserted. Total rows:", spark.table(T).count())

## 4. Prove it — evolution is **per-file** and **metadata-only**

`<table>.files` carries a `spec_id` per data file. After evolving you'll see **two specs**: old
files keep **`spec_id = 0`** (`days(ts)` only), new files use **`spec_id = 1`**
(`days(ts)` + `bucket(8, customer_id)`). **No old file was rewritten** — `table_health` shows the
data-file count rose only by the newly written files.

In [ ]:
print("data files by spec_id (0 = old days(ts) only; 1 = evolved days(ts)+bucket):")
spark.sql(f"""
    SELECT spec_id, COUNT(*) AS files, SUM(record_count) AS rows
    FROM {T}.files GROUP BY spec_id ORDER BY spec_id
""").show()

n_specs = spark.sql(f"SELECT COUNT(DISTINCT spec_id) AS n FROM {T}.files").first()["n"]
print(f"distinct spec_ids in the table's files: {n_specs}  (mixed specs coexist; reads work across them)")
print(f"data files: {health_before['data_files']} (before evolve) -> {health_after['data_files']} (after) "
      f"= +{health_after['data_files'] - health_before['data_files']} new files only; old files untouched")
print("\nWant old data in the new layout? It's OPTIONAL — re-bin spec_id=0 files with:")
print(f"  CALL iceberg_catalog.system.rewrite_data_files(table => 'default.lak7_events')   # LAK-2 procedure")

## Takeaways & "in real production…"

- **Choose transforms to match query patterns:** time filters -> `days`/`hours`(`ts`); lookups by a
  high-cardinality id -> `bucket(N, id)`; range/prefix scans -> `truncate`. (`years`/`months` and
  `identity` round out the set.) The transform *is* the partitioning decision.
- **Hidden partitioning => no CAST/derived columns.** Filter on the natural column (`ts`); the
  engine prunes. This is the cure for the pruning-defeating `CAST`/UDF filter in **SPK-7**.
- **Partition evolution is metadata-only and per-file.** Old data keeps its old spec and reads
  correctly; only new data uses the new spec. Rewriting old data into the new spec
  (`rewrite_data_files`) is **optional** — do it only when old partitions actually hurt a query.
- **Relate it forward:** pruning here is the lakehouse twin of **SPK-7**; partition *granularity*
  is what drives per-partition write cost in **LAK-8** (a 1-row MERGE rewriting a whole CoW
  partition). Keep partitions neither tiny-and-many nor giant; prefer `bucket` over raw
  high-cardinality columns; evolve deliberately and document the spec change.

## Teardown

Drop the table. `make clean` removes everything under `.tmp/` for a fully fresh warehouse.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {T}")
print("Dropped", T + ".", "(`make clean` clears all of .tmp.)")